## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

## Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_order_items")

## Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Remove invalid prices

In [0]:
df = df.filter(col("price") >= 0)

In [0]:
df = df.filter(col("shipping_charges") >= 0)

### Remove duplicates

In [0]:
df = df.dropDuplicates(["order_id", "product_id", "seller_id"])

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

## Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_order_items")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_order_items LIMIT 10